# 🚀 LLMEmbed: Rethinking Lightweight LLM's Genuine Function in Text Classification (ACL 2024)

Official Paper: *https://aclanthology.org/2024.acl-long.433*

This notebook reproduces the LLMEmbed pipeline on the SST-2 benchmark:
1. Extract embeddings from **BERT-large** (`bert-large-uncased`, 1024-d pooler output)
2. Extract embeddings from **RoBERTa-large** (`roberta-large`, 1024-d [CLS] output)
3. Extract embeddings from **LLaMA-2-7B** (`daryl149/llama-2-7b-chat-hf`, top-5 layers mean pooled, 5x4096-d)
4. Fuse embeddings with **Co-occurrence Pooling** (`DownstreamModel`)
5. Train & Evaluate on SST-2 (Accuracy & F1 Score)

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GB')

In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets accelerate scikit-learn tqdm

In [ ]:
# 3. Load SST-2 benchmark dataset
from datasets import load_dataset
try:
    dataset = load_dataset('stanfordnlp/sst2')
except Exception:
    dataset = load_dataset('nyu-mll/glue', 'sst2')
print(dataset)
print('Train samples:', len(dataset['train']))
print('Validation (test) samples:', len(dataset['validation']))

In [ ]:
# 4. Step 1: Extract BERT-large Embeddings
import os, torch, gc
from transformers import BertTokenizer, BertModel
from tqdm import trange

def extract_bert(sents, labels, mode, task='sst2'):
    model_path = 'bert-large-uncased'
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertModel.from_pretrained(model_path).cuda()
    model.eval()
    
    max_len, step = 128, 128
    sents_reps = []
    for idx in trange(0, len(sents), step, desc=f'BERT {mode}'):
        batch = sents[idx:idx+step]
        enc = tokenizer(batch, return_tensors='pt', max_length=max_len, padding='max_length', truncation=True).to('cuda')
        with torch.no_grad():
            out = model(**enc)
            sents_reps.append(out.pooler_output.cpu())
    sents_reps = torch.cat(sents_reps)
    labels_t = torch.tensor(labels)
    
    path = f'bert_embedding/{task}/dataset_tensor/'
    os.makedirs(path, exist_ok=True)
    torch.save(sents_reps, path + f'{mode}_sents.pt')
    torch.save(labels_t, path + f'{mode}_labels.pt')
    print(f'  Saved {mode}: sents={sents_reps.shape}, labels={labels_t.shape}')
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return sents_reps.shape

print('Extracting BERT-large embeddings...')
extract_bert(dataset['train']['sentence'], dataset['train']['label'], 'train')
extract_bert(dataset['validation']['sentence'], dataset['validation']['label'], 'test')
print('BERT-large extraction finished.')

In [ ]:
# 5. Step 2: Extract RoBERTa-large Embeddings
import os, torch, gc
from transformers import RobertaTokenizer, RobertaModel
from tqdm import trange

def extract_roberta(sents, labels, mode, task='sst2'):
    model_path = 'roberta-large'
    tokenizer = RobertaTokenizer.from_pretrained(model_path)
    model = RobertaModel.from_pretrained(model_path).cuda()
    model.eval()
    
    max_len, step = 128, 128
    sents_reps = []
    for idx in trange(0, len(sents), step, desc=f'RoBERTa {mode}'):
        batch = sents[idx:idx+step]
        enc = tokenizer(batch, return_tensors='pt', max_length=max_len, padding='max_length', truncation=True).to('cuda')
        with torch.no_grad():
            out = model(**enc)
            sents_reps.append(out.last_hidden_state[:, 0, :].cpu())
    sents_reps = torch.cat(sents_reps)
    labels_t = torch.tensor(labels)
    
    path = f'roberta_embedding/{task}/dataset_tensor/'
    os.makedirs(path, exist_ok=True)
    torch.save(sents_reps, path + f'{mode}_sents.pt')
    torch.save(labels_t, path + f'{mode}_labels.pt')
    print(f'  Saved {mode}: sents={sents_reps.shape}, labels={labels_t.shape}')
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return sents_reps.shape

print('Extracting RoBERTa-large embeddings...')
extract_roberta(dataset['train']['sentence'], dataset['train']['label'], 'train')
extract_roberta(dataset['validation']['sentence'], dataset['validation']['label'], 'test')
print('RoBERTa-large extraction finished.')

In [ ]:
# 6. Step 3: Extract LLaMA-2-7B Embeddings
import os, torch, gc
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from tqdm import trange

def extract_llama(sents, labels, mode, task='sst2', max_len=128, step=32):
    model_id = 'daryl149/llama-2-7b-chat-hf'
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    
    config = AutoConfig.from_pretrained(model_id, output_hidden_states=True, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, config=config, device_map='auto',
        torch_dtype=torch.float16, trust_remote_code=True
    )
    model.eval()
    
    sents_reps = []
    for idx in trange(0, len(sents), step, desc=f'LLaMA {mode}'):
        batch = sents[idx:idx+step]
        enc = tokenizer(batch, return_tensors='pt', max_length=max_len, padding='max_length', truncation=True)
        enc = enc.to(model.device)
        with torch.no_grad():
            out = model(**enc)
            reps_5L = []
            for layer in range(-1, -6, -1):
                reps_5L.append(torch.mean(out.hidden_states[layer], axis=1))
            reps_5L = torch.stack(reps_5L, axis=1)
        sents_reps.append(reps_5L.cpu())
    sents_reps = torch.cat(sents_reps)
    labels_t = torch.tensor(labels)
    
    path = f'llama2_embedding/{task}/dataset_tensor/'
    os.makedirs(path, exist_ok=True)
    torch.save(sents_reps, path + f'{mode}_sents.pt')
    torch.save(labels_t, path + f'{mode}_labels.pt')
    print(f'  Saved {mode}: sents={sents_reps.shape}, labels={labels_t.shape}')
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return sents_reps.shape

print('Extracting LLaMA-2-7B embeddings...')
extract_llama(dataset['train']['sentence'], dataset['train']['label'], 'train', max_len=128, step=32)
extract_llama(dataset['validation']['sentence'], dataset['validation']['label'], 'test', max_len=128, step=32)
print('LLaMA-2-7B extraction finished.')

In [ ]:
# 7. Step 4: Verify Extracted Tensors
import torch, os
for folder in ['llama2_embedding/sst2/dataset_tensor', 'bert_embedding/sst2/dataset_tensor', 'roberta_embedding/sst2/dataset_tensor']:
    print(f'=== {folder} ===')
    if os.path.exists(folder):
        for f in sorted(os.listdir(folder)):
            t = torch.load(os.path.join(folder, f), weights_only=False)
            print(f'  {f}: shape={t.shape}, dtype={t.dtype}')
    else:
        print('  NOT FOUND!')

In [ ]:
# 8. Step 5: DownstreamModel Architecture (Author's exact implementation)
import torch
import torch.nn as nn
import torch.nn.functional as F

class DownstreamModel(nn.Module):
    def __init__(self, class_num, SIGMA):
        super(DownstreamModel, self).__init__()
        self.SIGMA = SIGMA
        self.compress_layers = nn.ModuleList()
        for _ in range(5):
            layers = []
            layers.append(nn.Linear(4096, 1024))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.5))
            self.compress_layers.append(nn.Sequential(*layers))
        
        self.fc1 = nn.Linear(4145, 1024)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(256, class_num)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, input_l, input_b, input_r):
        batch_size = input_l.shape[0]
        split_tensors = torch.split(input_l, 1, dim=1)
        input = []
        for i, split_tensor in enumerate(split_tensors):
            split_tensor = split_tensor.reshape(batch_size,-1)
            input.append(self.compress_layers[i](split_tensor))
        input.append(input_b)
        input.append(input_r)
        input = torch.stack(input, dim=1)
        input_T = input.transpose(1, 2)
        input_P = torch.matmul(input, input_T)
        input_P = input_P.reshape(batch_size, -1)
        input_P = 2*F.sigmoid(self.SIGMA * input_P) - 1
        
        a = torch.mean(input_l, dim=1)
        input = torch.cat([input_P, a], dim=1)
        
        output = self.fc1(input)
        output = self.relu1(output)
        output = self.dropout1(output)
        output = self.fc2(output)
        output = self.relu2(output)
        output = self.dropout2(output)
        output = self.fc3(output)
        output = self.softmax(output)
        return output

print('DownstreamModel successfully initialized.')

In [ ]:
# 9. Step 6: Dataset & Training Functions (Author's exact implementation)
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

class MyDataset(Dataset):
    def __init__(self, mode, l_path, b_path, r_path):
        self.l_sents_reps = torch.load(l_path + f'{mode}_sents.pt', weights_only=False)
        self.b_sents_reps = torch.load(b_path + f'{mode}_sents.pt', weights_only=False)
        self.r_sents_reps = torch.load(r_path + f'{mode}_sents.pt', weights_only=False)
        self.labels = torch.load(l_path + f'{mode}_labels.pt', weights_only=False)
        self.sample_num = self.labels.shape[0]
    def __getitem__(self, index):
        return self.l_sents_reps[index], self.b_sents_reps[index], self.r_sents_reps[index], self.labels[index]
    def __len__(self):
        return self.sample_num

def Train(dataloader, device, model, loss_fn, optimizer):
    loss_list, acc_list, f1_list = [], [], []
    for batch_i, batch_loader in enumerate(tqdm(dataloader, desc='Train')):
        batch_l, batch_b, batch_r, batch_y = batch_loader
        batch_l, batch_b, batch_r, batch_y = batch_l.to(device), batch_b.to(device), batch_r.to(device), batch_y.to(device)
        model.train()
        pred = model(batch_l.float(), batch_b.float(), batch_r.float())
        loss = loss_fn(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pred_y = torch.max(pred, 1).indices
        acc = accuracy_score(batch_y.cpu(), pred_y.cpu())
        f1 = f1_score(batch_y.cpu(), pred_y.cpu())
        loss_list.append(loss.item())
        acc_list.append(acc)
        f1_list.append(f1)
    print(f'  loss: {np.mean(loss_list):.4f}, acc: {np.mean(acc_list):.4f}, F1: {np.mean(f1_list):.4f}')

def Test(dataloader, device, model, loss_fn):
    avg_loss = 0
    total_pred, total_y = [], []
    for batch_i, batch_loader in enumerate(tqdm(dataloader, desc='Test')):
        batch_l, batch_b, batch_r, batch_y = batch_loader
        batch_l, batch_b, batch_r, batch_y = batch_l.to(device), batch_b.to(device), batch_r.to(device), batch_y.to(device)
        model.eval()
        with torch.no_grad():
            pred = model(batch_l.float(), batch_b.float(), batch_r.float())
            loss = loss_fn(pred, batch_y)
            avg_loss += loss.item()
        pred_y = torch.max(pred, 1).indices
        total_pred.append(pred_y.cpu())
        total_y.append(batch_y.cpu())
    avg_loss = avg_loss / (batch_i+1)
    total_y = torch.cat(total_y)
    total_pred = torch.cat(total_pred)
    acc = accuracy_score(total_y, total_pred)
    f1 = f1_score(total_y, total_pred)
    print(f'  avg loss: {avg_loss:.4f}')
    print(f'  acc: {acc:.4f}')
    print(f'  F1 Score: {f1:.4f}')
    return acc, f1

print('MyDataset, Train, and Test functions successfully defined.')

In [ ]:
# 10. Step 7: Train & Evaluate LLMEmbed Downstream Classifier
# Author's hyperparameters: task=sst2, epoches=10, SIGMA=0.1, batch_size=1024, lr=1e-4
import torch
import torch.nn as nn
import torch.optim as optim

task = 'sst2'
device = 'cuda:0'
epoches = 10
SIGMA = 0.1
batch_size = 1024
lr = 1e-4
class_num = 2

l_dataset_path = f'llama2_embedding/{task}/dataset_tensor/'
b_dataset_path = f'bert_embedding/{task}/dataset_tensor/'
r_dataset_path = f'roberta_embedding/{task}/dataset_tensor/'

train_data = MyDataset('train', l_dataset_path, b_dataset_path, r_dataset_path)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_data = MyDataset('test', l_dataset_path, b_dataset_path, r_dataset_path)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

model = DownstreamModel(class_num, SIGMA).to(device)
loss_fn = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr)

print('Training DownstreamModel for 10 epochs...')
for epoch in range(epoches):
    print(f'--------------------------- epoch {epoch} ---------------------------')
    Train(train_loader, device, model, loss_fn, optimizer)

print()
print('Evaluating DownstreamModel on Test (Validation) set...')
acc, f1 = Test(test_loader, device, model, loss_fn)

In [ ]:
# 11. Final Results & Comparison with ACL 2024 Paper (Table 2)
print('=' * 65)
print('ACL 2024 Paper Benchmark Results on SST-2 (Table 2):')
print('  - BERT-large (fine-tuned):        0.8761')
print('  - RoBERTa-large (fine-tuned):     0.9025')
print('  - LLaMA2 7B (IO prompt):          0.8922')
print('  - CARP (GPT-3 175B):              0.9569')
print('  - LLMEmbed (Ours paper target):   0.9553')
print('=' * 65)
print(f'OUR KAGGLE REPRODUCTION RESULT: Accuracy = {acc:.4f}, F1 Score = {f1:.4f}')
print('=' * 65)